<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">3c - Workshop (AI Extensions) - Open Source Decision Intelligence</span>
</div>

Decision Intelligence applied in this module:  
* Listing key factors to consider when making a sound decision  
* Using OSS (open-source) reasoning models to optimize the decision approach 
* Decision Scenario: Use a decision framework (Ben Franklin's Pro & Con List) to create a decision plan  
* Improving Decision Intelligence process by explicitly providing decision frameworks and additional context 

A recommended enterprise pattern is to scale Articial Intelligence strategy with three key pillars:
* **Commercial AI** (OpenAI and other proprietary Generative AI providers)
* **Open Source (OpenWeight) AI** (open-source AI providers)
* **Vendor and Partner AI** (i.e. company HR Software, contract software) 

> 📝 Note: There is a technical difference between Open Source and Open Weight models. Open Weight models generally only provide you the weights & biases file(s) with a friendly open source license. Open Source models include the pre/post training data, training scripts and sometimes the training checkpoints. Those three additions allow the models to be fully retrained by anyone. More info can be found here: https://opensource.org/ai/open-weights. For this workshop, we will use the term open source more loosely. A great majority of the cases most organizations just want two things: the weights & biases file(s) and a friendly licnese. Certain organizations certainly require full model assets for risk management. While the full open source model assets are nice to have, they offer minimal value in most cases. 

These three pillars (listed above) strategically form AI capability and capacity in what I like to refer to as the **"Generative AI Brain"**. This is illustrated below with sample providers. Most organizations scale their AI investments with Commercial AI providers, such as: OpenAI, Anthropic or Google. However, some organizations prefer to have more governance controls over their AI and prefer open-source alternatives. For hobby consumers, open source translates to no credit card requirements for APIs and very high privacy. 
Note that there are some commercial providers like Meta or Cohere that provide open weight models. Finally in the third tier, an organization may have an existing relationship with a vendor/partner that has their own AI integrations. For example, Adobe offers enterprise graphic design/contract software and leveraging their built-in Generative AI capabilities rather building their own can make a lot of sense. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/AIBrainPillars.png">

Most developer AI frameworks work across all the pillars mentioned above. It allows all types of models (commercial or proprietary) and almost any APIs to be orchestrated to faciliate enterprise Decision Intelligence. This is a much deeper subject and for simplicity this module highlights how to use local GenAI models (i.e. LLMs) with Microsoft AI for decision-making. Models from the OpenAI OSS family will be selected to run locally using LMStudio as a local REST endpoint that will interface with Microsoft AI orchestration.  

Open Source (OSS) models vary dramatically in size. They can have a small number of parameters/activated layers and perform great locally on your mobile device! They can also have a huge number parameters that rivals commercial AI LLMs. 

In the cases where OSS models have a small number of parameters, they are are considered SLMs (Small Language Models) with parameters generally below the 30 Billion parameter threshhold. This allows most of these models to run comfortably on commodity hardware available to personal users. This doesn't mean these models are not capable of performing well. While these models certainly may lack the general knowledge breadth of frontier AI Large Language Models, SLMs make up for it by providing very specialized logic, math and reasoning capabilities. For example, an SLM only trained for a specific language (English) and a certain domain (legal industry) can include only the relevant training information. Therefore, it can be offered as a much more compact model with many less parameters. 

In the cases where OSS models have a large number of parameters, they can perform just as well as commercial models! OSS models with large parameters require enterprise commmerical hardware to execute at scale. Below is an image from an independent AI benchmarking site ArtificialAnalysis.ai across proprietary and OSS models. Notice that almost half of the Top 30 performing models are open weight models (blue): 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/ArtificialAnalysis-ModelBenchmark-202604.png">  

For this workshop, we will be working primarily with SLM family of OSS models, because not everyone has H100 or B200 Nvidia GPUs to deploy huge parameter models. However, as you will see these smaller models are quite good. As of April 2026, OpenAI's gpt-oss-20 or Gemma 4 open-source reasoning models performs about 2 generations back of frontier LLM performance. Keep in mind these are general 20B & 26B parameter OSS models. This model can be fine-tuned or mid-trained and can perform much better. For current LM statistics, please visit: https://llm-stats.com/leaderboards/open-llm-leaderboard. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/LMStats-Benchmark-202604.png">  

----
### Step 1 - Get Started with LMStudio and Local Open Source AI Models 

Steps to get started:
* Download & install the latest LMStudio version: https://lmstudio.ai/ (Windows, Mac or Linux) 
* Run the LMStudio studio application.
* In the LMStudio application, search for **Gemma 4 26B A4B** GGUF (MLX is optimized on macOS) in the "Discover" section of LMStudio. A variety of Gemma options that are official and unofficial from hobbyists will appear. Typically, selecting the official model with the most downloads will provide the best results. In this case, you can be safe by selecting the official **Google** provider. You can select different quantizations of the model, to optimize the performance. 
* In the experiment below, the **26B parameter model is being used with 4bit quantization** is selected. LMStudio will inspect your hardware and let you know which quantized version of the model(s) is optimal for your hardware. Note: Even computers with small graphics cards can run these models well locally. Furthermore, laptops such as the Macbook Pro with Neural Engine can run LMStudio local models as well. You just need memory. 
* Start the LMStudio Server with the **Gemma 4 26B A4B** model loaded. This will start a local REST endpoint with a URI similar to http://10.0.0.18:1234/v1 
* The LMStudio local server does not have default security, you can simply check by navigating to this link in any browser to check if a model is loaded: **http://10.0.0.18:1234/v1/models** in the web browser. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/LMStudioServer.png"> 

----
### Step 2 - Initialize ChatClient using OpenAI libraries

Execute the next cell to:
* Use the Configuration Builder to use the local LMStudio Server  
* Use the local API configuration to build an API compatible OpenAIClient
* The API Compatible OpenAIClient can be converted to a Microsoft.Extensions.AI ChatClient abtraction
* Note: Notice there is no security being passed in and it is simply a URL

In [1]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 10.0.6"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.6"
#r "nuget: System.Text.Json, 10.0.6"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var localAIEndpoint = config["LocalAI:Endpoint"]; // local LLMs do not require an API key
var localAIAPIKey = config["LocalAI:APIKey"]; // this can be localhost or an IP address
var localAIModelDeploymentName = config["LocalAI:ModelDeploymentName"]; // Local AI Model Name. Another Option: "openai/gpt-oss-20b";

// Display the loaded configuration settings to verify they were loaded correctly
Console.WriteLine($"Local AI Endpoint: {localAIEndpoint}"); 
Console.WriteLine($"Local AI API Key: {localAIAPIKey}");    
Console.WriteLine($"Local AI Model Deployment Name: {localAIModelDeploymentName}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.6 Microsoft.Extensions.Configuration.Json, 10.0.6 System.Text.Json, 10.0.6

Local AI Endpoint: http://10.0.0.61:1234/v1/
Local AI API Key: not_needed_for_lmstudio_ollama_etc
Local AI Model Deployment Name: google/gemma-4-26b-a4b


In [2]:
// Install the required AI packages
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.6"
#r "nuget: Microsoft.Extensions.AI, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
#r "nuget: OpenAI, 2.10.0"

using Microsoft.Extensions.AI;
using OpenAI;
using System.ClientModel; // used by APiKeyCredential class

var apiCredentials = new ApiKeyCredential(localAIAPIKey);
var openAIClientOptions = new OpenAIClientOptions
{
    Endpoint = new Uri(localAIEndpoint)
};

// Create a local AI client 
var localAIClient = new OpenAIClient(apiCredentials, openAIClientOptions);

// Wrap the OpenAI-compatible chat client with the Microsoft.Extensions.AI abstraction.
IChatClient localAIChatClient = localAIClient
    .GetChatClient(localAIModelDeploymentName)
    .AsIChatClient();

Installed Packages Microsoft.Extensions.AI, 10.5.0 Microsoft.Extensions.AI.Abstractions, 10.5.0 Microsoft.Extensions.AI.OpenAI, 10.5.0 Microsoft.Extensions.DependencyInjection, 10.0.6 OpenAI, 2.10.0

---
### Step 3 - Open Source AI with Decision Intelligence 

The OpenAI .NET library allows one to interact with any API service that adheres to the OpenAI specifications. This can be the ChatCompletions API or the newer Responses API. Notice the method to add LMStudio capability was simply enabled via the **AddOpenAIChatCompletion** method above. 

Note:
* OpenAI Prompt Execution Settings (Temperature, TopK) are the same in LMStudio as they are for OpenAI and Azure OpenAI
* OSS models have specific model cards identifying best practice configuration settings 

Execute the cell below about decision factors for a investment property. 

In [3]:
// Define the system prompt for the Decision Intelligence assistant
// Note: For an enterprise-grade Decision Intelligence assistant, you would want to provide more detailed instructions and guidelines. 
var systemDecisionPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

-------------------------------
Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Provide detailed decision-making criteria for evaluating the investment decision
var simpleDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

What key factors should you evaluate to ensure a sound investment decision, including financial, 
market, and property-specific considerations? 
Outline the critical steps and criteria for assessing location, potential rental income, 
financing options, long-term property value, and associated risks. 
""";

List<ChatMessage> chatMessages =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, simpleDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryResponse = await localAIChatClient.GetResponseAsync(chatMessages, chatOptions);
var chatHistoryResponseText = chatHistoryResponse.Text;

// Display the response string as Markdown
chatHistoryResponseText.DisplayAs("text/markdown");

| Category | Key Factor | Evaluation Criteria & Critical Steps | Impact on Decision |
| :--- | :--- | :--- | :--- |
| Financial | Cash Flow Analysis | Calculate Net Operating Income (NOI) by subtracting all operating expenses (taxes, insurance, maintenance, property management) from gross rental income. | Determines if the property generates monthly profit or requires out-of-pocket subsidies. |
| Financial | Return on Investment (ROI) | Calculate Cash-on-Cash Return and Cap Rate. Compare these against alternative investments like stocks or bonds. | Provides a standardized metric to compare the property's performance against other asset classes. |
| Financial | Financing Structure | Evaluate down payment requirements (typically higher for second homes), interest rates, and debt-to-income ratios. | Affects initial capital outlay and monthly liquidity requirements. |
| Market | Location Intelligence | Analyze proximity to amenities (schools, transit, employment hubs), crime rates, and local economic drivers. | Primary driver of both rental demand and long-term capital appreciation. |
| Market | Rental Demand & Seasonality | Study vacancy rates, seasonal fluctuations in tourism (if applicable), and local population growth trends. | Determines the stability of income and the frequency of turnover/vacancy costs. |
| Market | Supply & Regulatory Environment | Check zoning laws, short-term rental (Airbnb) regulations, and future planned developments in the area. | Mitigates the risk of sudden legislative changes that could render the business model illegal. |
| Property-Specific | Condition & Capital Expenditures (CapEx) | Perform professional inspections focusing on structural integrity, roof, HVAC, and plumbing. Create a sinking fund for major repairs. | High upfront maintenance costs can wipe out annual cash flow entirely. |
| Property-Specific | Scalability & Versatility | Assess if the property can be easily renovated to increase value or adapted for different tenant types (e.g., adding a bedroom). | Offers "forced appreciation" opportunities to boost equity. |
| Risk Management | Market Volatility & Liquidity | Assess how easily the property can be sold in a downturn and the risk of local market saturation. | High-value second homes are often less liquid than primary residences. |
| Risk Management | Diversification & Correlation | Evaluate how this investment correlates with your existing portfolio and primary residence. | Prevents over-exposure to a single geographic region or economic sector. |

---
### Step 4 - Open Source AI with Decision Intelligence (Advanced)

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-WillRodgers-RealEstate.png"> 

Advanced Prompt Engineering techniques can be applied to OSS (open-source) models as well. In the example below a more advanced reasoning decision prompt will be used to provide additional instructions to the GenAI model. Reasoning models do a nice job in approaching the problem with an inner monologue, however you can provide additional instructions for the model to consider as they are thinking about an approach.  

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Use Chain of Thought to prompt the OSS model
// Use the Minto Pyramid to communicate the decision 
var advancedDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

Before providing any answer, in your reasoning process consider the following:
Understand the Problem: Carefully read and understand the user's question or request. 
Break Down the Reasoning Process: Outline the steps required to solve the problem or respond to the request logically and sequentially. Think aloud and describe each step in detail. 
Always aim to make your thought process transparent and logical. 
Explain Each Step: Provide reasoning or calculations for each step, explaining how you arrive at each part of your answer. 
Provide structured, logical, and comprehensive advice. 
Arrive at the Final Answer: Only after completing all steps, provide the final answer or solution. 
Review the Thought Process: Double-check the reasoning for errors or gaps before finalizing your response. 

Communicate the final decision using the Minto Pyramid Principle.
""";

List<ChatMessage> chatMessagesDecision =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, advancedDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64,
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryAdvancedDecisionPromptResponse = await localAIChatClient.GetResponseAsync(chatMessagesDecision, chatOptions);
var chatHistoryAdvancedDecisionPromptText = chatHistoryAdvancedDecisionPromptResponse.Text;

// Render the response string as Markdown
chatHistoryAdvancedDecisionPromptText.DisplayAs("text/markdown");

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatMessages.Add(new ChatMessage(ChatRole.Assistant,chatHistoryAdvancedDecisionPromptText));

### Reasoning Process

To evaluate the purchase of a secondary home as an investment property, I will follow this analytical framework:

1. **Define Objective and Parameters**: Establish whether the goal is wealth preservation, passive income generation, or capital appreciation, as the optimal strategy depends on these distinct drivers.
2. **Identify Key Decision Variables**: Isolate the critical factors including market volatility, interest rates, tax implications (depreciation vs. capital gains), liquidity, and operational overhead (management/maintenance).
3. **Develop a Comparative Framework**: Create a multi-dimensional evaluation matrix to weigh different investment archetypes (e.g., Long-term Rental, Short-term Vacation Rental, or Fix-and-Flip).
4. **Conduct Trade-off Analysis**: Analyze the tension between risk and reward (e.g., high-yield/high-effort vs. low-yield/low-effort) and the impact of macro-economic uncertainty on property valuation.
5. **Synthesize via Minto Pyramid Principle**: Structure the final output starting with a core recommendation, supported by logical groupings of arguments (Financials, Operations, Risks), which are further detailed by specific data points.

### Evaluation Framework for Secondary Home Investment

| Dimension | Strategic Consideration | Key Trade-offs & Uncertainties | Critical Metrics to Monitor |
| :--- | :--- | :--- | :--- |
| **Core Strategy** | Determine primary driver: Cash Flow (Income) vs. Appreciation (Growth). | High cash flow often requires high management intensity; high appreciation involves higher market risk and lower liquidity. | Cap Rate, Cash-on-Cash Return, Projected Annual Appreciation. |
| **Financial Structure** | Debt-to-Equity ratio and financing terms for non-primary residences. | Secondary home loans typically require higher down payments (20-25%) and carry higher interest rates than primary residences. | Debt Service Coverage Ratio (DSCR), Interest Rate Spread, Loan-to-Value (LTV). |
| **Operational Model** | Self-management vs. Professional Property Management. | Self-management maximizes margins but increases "opportunity cost" of your time; management provides scalability but eats into Net Operating Income (NOI). | Operating Expense Ratio (OER), Vacancy Rate, Management Fee %. |
| **Tax & Legal** | Tax treatment of depreciation, 1031 exchanges, and local STR (Short-Term Rental) regulations. | Short-term rentals may offer higher gross revenue but are subject to volatile municipal zoning laws and higher occupancy taxes. | Effective Tax Rate, Depreciation Schedule, Local Regulatory Stability. |
| **Risk Profile** | Sensitivity to interest rate fluctuations and macroeconomic downturns. | Real estate is a "lumpy" asset; exiting during a market downturn is difficult due to high transaction costs (closing fees/commissions). | Break-even Occupancy, Sensitivity Analysis (e.g., 10% drop in rents). |

---
### Step 5 - Open Source AI with The Ben Franklin Decision Framework 

> 📜 **_"By failing to prepare, you are preparing to fail."_**
>
> -- <cite>Ben Franklin (Founding Father of the United States, inventor, godfather of Decision Science)</cite> 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/Articles/main/20230326-Make-Great-Decisions-Using-Ben-Franklins-Pros-And-Cons-Method/Image-BenFranklinDecisionMakingMethod.png">

#### Tom Brady's use of a Decision Framework

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-TomBrady-ProsAndConsList.png">

Tom Brady's decision to join the Tampa Bay Buccaneers in 2020 marked a significant in his legendary NFL career. After 20 seasons and six Super Bowl championships with the New England Patriots, Brady became a free agent and chose to sign with the Bucs. How did he arrive at this decision? On the Fox broadcast on 09.29.2024, while covering the Buccaneers vs Philadelphia Eagles game, Tom Brady described how he arrived at this decision.

In the screenshot below, Tom Brady is holding up some small paper cards he is showing the audience of the broadcast. Brady mentioned he wrote down the personal decision criteria that was important and how each team compared in that criteria (salary, weather etc). He used this to select the Tampa Bay Buccaneers as his team, where he went on to win a Super Bowl in his first year there! **After 250 years since it's inception, Tom Brady used the "Ben Franklin Decision Framework" to decide where to play NFL quaterback!!**  

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/OpenSourceDecisionIntelligence-BenFranklinDecisionFramework-TomBrady.png">

#### Steps for Ben Franklin's Decision Framework

Below are the steps Ben Franklin recommends when making a decision, which he called his "Decision Making Method of Moral Algebra":  
- Frame a decision that has two options (Yes or a No)
- Divide an area into two competing halves: a "Pro" side and "Con" side
- Label the top of one side "Pro" (for) and the other "Con" (against)
- Under each respective side, over a period of time (Ben Franklin recommended days, this could be minutes) write down various reasons/arguments that support (Pro) or are against (Con) the decision
- After spending some time thinking exhaustively and writing down the reasons, weight the different Pro and Con reasons/arguments
- Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
- The side with the most remaining reasons is the option one should select for the decision in question

Learn more about Ben Franklin's Decision Framework: https://medium.com/@bartczernicki/make-great-decisions-using-ben-franklins-decision-making-method-c7fb8b17905c  

#### Decision Scenario - Should a Family Decide to Take a Luxury Vacation?

Should a family take a luxury family vacation this year? Just like Brady mapped out whether joining the Bucs would satisfy his key personal and professional goals, you can list the factors that matter most for your family—budget, timing, destination climate, activities for the kids—and lay them out on your own “decision cards.” Weigh each component carefully, just as Brady weighed his NFL future. Because if it worked to land Brady in Tampa (where he won yet another Super Bowl), imagine what it can do for a family’s dream getaway.

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-Vacation.png">

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

var benFranklinLuxuryVacationDecisionPrompt = """
Apply the Ben Franklin Decision-Making Framework (Pro and Con list) to evaluate whether or not to take a luxury family vacation. 
List at most 5 pros and at most 5 cons to help the user make an informed decision.
""";

List<ChatMessage> chatMessagesLuxuryVacationDecision =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, benFranklinLuxuryVacationDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryAdvancedDecisionPromptResponse = await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecision, chatOptions);
var chatHistoryAdvancedDecisionPromptText = chatHistoryAdvancedDecisionPromptResponse.Text;

// Render the response string as Markdown
chatHistoryAdvancedDecisionPromptText.DisplayAs("text/markdown");

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatMessages.Add(new ChatMessage(ChatRole.Assistant,chatHistoryAdvancedDecisionPromptText));

| Category | Argument Type | Factor | Description/Impact |
| :--- | :--- | :--- | :--- |
| **Pros (Advantages)** | Emotional | Family Bonding | Creates high-quality, undistracted time to strengthen relationships and build lasting memories. |
| | Psychological | Mental Reset | Provides a significant reduction in burnout and stress through complete detachment from daily responsibilities. |
| | Experiential | Cultural Enrichment | Exposure to new environments, cuisines, and perspectives that foster personal growth for all family members. |
| | Social | Relationship Capital | Serves as a "recharge" event that can improve morale and productivity upon returning to work/school routines. |
| | Reward | Celebration of Milestones | Acts as a tangible reward for past hard work or significant life achievements, reinforcing positive reinforcement loops. |
| **Cons (Disadvantages)** | Financial | Opportunity Cost | Significant capital outflow that could otherwise be allocated to long-term investments, debt reduction, or emergency funds. |
| | Operational | Logistical Complexity | High degree of planning, scheduling, and coordination required, which can induce "pre-vacation" stress. |
| | Financial | Post-Trip Regret | The risk of "lifestyle creep" or the psychological deflation felt when returning to a more mundane reality after luxury. |
| | Risk/Uncertainty | Unforeseen Expenses | Vulnerability to unexpected costs (travel delays, medical needs, price fluctuations) that exceed the initial budget. |
| | Strategic | Resource Depletion | Potential depletion of "slack" in the family's annual budget, reducing the ability to respond to future urgent needs. |

#### Improving the Ben Franklin's Decision Framework with Local AI

For those familiar with the Ben Franklin decision framework, the output from the AI model above may not be exactly what most would anticipate. The Ben Franklin framework could be partially understood by the AI process nor fully applied. Open-Source GenAI models that have a small amount of parameters (< ~27 billion parameters) may not have all the inherent Decision Intelligence knowledge "trained" into the model. The exception being domain-specific models that are specifically trained on data sets for that domain. These domain-specific models can fill their "limited knowledge" with information that is pertinent to the tasks, while maintaining a small amount of parameters. Therefore, you could train small AI models that specialize in Decision Intelligence. 

One simple way to improve the outcome is to provide the explicit steps of the "Ben Franklin Decision Framework" into the prompt context. This basically provides the instructions of the decision framework directly to the model; regardless if the GenAI model was trained with decision framework data. By doing this extra explicit step, there is no ambiguity for the AI model how to approach the decision process. 

In the example below, the prompt context is provided with the Ben Franklin Decision Framework steps. Contrast this with the example above, where the decision recommendation is not clear. 

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option. If there less than 5 reasons, only list the actual number of reasons there are for each side. Do not list filler reasons to reach 5 if there are not actually 5 reasons for that side.
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question
IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Answer the Question: Should I take a luxury family vacation?
""";

List<ChatMessage> chatMessagesLuxuryVacationDecisionImproved =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, explicitBenFranklinDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedResponse = await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImproved, chatOptions);
var chatLuxuryVacationDecisionImprovedResponseText = chatLuxuryVacationDecisionImprovedResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedResponseText.DisplayAs("text/markdown");

| Decision Framework Step | Analysis Details |
| :--- | :--- |
| 1) Framed Decision | Should I take a luxury family vacation? (Yes or No) |
| 2, 3, 4) Pro vs Con Arguments | **Pro:** <br>1. Strengthening family bonds <br>2. Creating lasting lifelong memories <br>3. Stress reduction and mental rejuvenation <br><br> **Con:** <br>1. Significant financial strain/cost <br>2. Logistical complexity and planning stress |
| 5, 6) Weighting & Importance | **Weighting Process:** <br>- The "Financial strain" (Con) is a heavy weight; it effectively cancels out the "Stress reduction" (Pro) because the cost may create new stresses. <br>- The "Creating lasting memories" and "Strengthening family bonds" (Pro) are high-value weights; they are of similar value and together outweigh the "Logistical complexity" (Con). |
| 7) Final Recommendation | **Decision: YES.** After weighing the arguments, the Pro side retains more remaining substantive points/value than the Con side. |

The GenAI model may or may not recommend a luxury vacation depending on the executed run. It's decision response is highly generic and isn't grounded on personal information that can influence the decision recommendation. This can be dramatically improved further! Imagine if the GenAI model had access to: your finances, current stress level, the last time you took a vacation, any upcoming major purchases, family dynamic?!  

In the optimized decision example below, additional context is provided with that information. Notice how it changes the the Pro and Con list.  

Just like Tom Brady, the AI could craft a Pro and Con list specific and personalized to your scenario!

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Try changing the background information to see how it affects the decision-making process
var gatheredIntelligenceFamilyBackground = """
GATHERED INTELLIGENCE FAMILY BACKGROUND USED AS ONLY DECISION DRIVERS: 
1) You are considering purchasing a secondary home as an investment property. 
2) You have been working long hours and have not taken a vacation in over a year. 
3) You have received a recent promotion (pay raise with a large bonus coming in a few months).
4) Your car is finishing its lease and will need to be replaced soon. 
5) The kids are about to start college soon. 
6) You have had some recent medical expenses that you are unsure insurance will cover.
""";

// Try to adjust the specificity of the decision-making criteria to see how it affects the decision-making process
var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option. If there less than 5 reasons, only list the actual number of reasons there are for each side. Do not list filler reasons to reach 5 if there are not actually 5 reasons for that side.
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question
IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Answer the Question: Should I take a luxury family vacation?
""";

List<ChatMessage> chatMessagesLuxuryVacationDecisionImprovedWithGatheredIntelligence =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.Assistant, gatheredIntelligenceFamilyBackground),
    new(ChatRole.User, explicitBenFranklinDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 0.2f,
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponse = 
    await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImprovedWithGatheredIntelligence, chatOptions);
var chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponseText = chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponseText.DisplayAs("text/markdown");

| Step | Analysis Process | Resulting Logic/Content |
| :--- | :--- | :--- |
| 1. Frame Decision | Define the binary choice. | Should I take a luxury family vacation? (Yes or No) |
| 2, 3, 4. Pros vs Cons | List arguments based on provided intelligence. | **Pro:**<br>1. Mitigates burnout from long working hours.<br>2. Leverages recent promotion/bonus potential.<br>3. Creates memories before kids start college.<br><br>**Con:**<br>1. High cost conflicts with uncertain medical expenses.<br>2. High cost conflicts with upcoming car replacement needs.<br>3. High cost conflicts with upcoming college tuition costs. |
| 5, 6. Weighting & Importance | Compare values and cross-reference arguments. | **Weighting Analysis:**<br>- The "Pro" side contains 1 high-weight reason (Burnout/Mental Health) and 2 medium-weight reasons (Timing/Bonus).<br>- The "Con" side contains 3 high-weight reasons (Medical uncertainty, Car replacement, College tuition).<br><br>**Cross-off Process:**<br>- Cross off Pro #1 (Burnout) against Con #1 (Medical Expenses) [High weight vs High weight].<br>- Cross off Pro #2 (Bonus) against Con #2 (Car Replacement) [Medium weight vs Medium weight].<br>- Cross off Pro #3 (College Timing) against Con #3 (College Costs) [Medium weight vs Medium weight]. |
| 7. Final Recommendation | Identify the side with the most remaining reasons. | **Decision: No.**<br>After weighing all arguments, the "Con" side retains more distinct, heavy-weight financial obligations that remain unaddressed by the "Pro" side's benefits. |

Notice how providing personal family background information changes the entire dynamic of the information used in the decision framework and how it influences the recommended decision. The decision process is more specific not only to the scenario, but also it provides contextual background information. This makes the decision process more personalized and potentially much more accurate! 

---
### Step 6 - Higher Reasoning Effort for High-Stakes Decisions  

Let's assume that this a very high-stakes decision. In this case we would prefer to have the AI system spend more time on the decision. Let's turn reasoning on to see how this changes the answer. 

In [ ]:
// Define reasoning options for the chat completion.
// Note: For additional reasoning capabilities, reasoning effort is set to High
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.High
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 0.2f, // Make the decision more deterministic based on the gathered intelligence background information 
    Reasoning = reasoningOptions
};

List<ChatMessage> chatMessagesLuxuryVacationDecisionImprovedWithGatheredIntelligence =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.Assistant, gatheredIntelligenceFamilyBackground),
    new(ChatRole.User, "<|think|> " + explicitBenFranklinDecisionPrompt),
];

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponse = 
    await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImprovedWithGatheredIntelligence, chatOptions);
var chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponseText = chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponse.Text;

// Extract the reasoning text from the chat response by filtering for TextReasoningContent and concatenating the text
string reasoningText = string.Join(
    Environment.NewLine + Environment.NewLine,
    chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponse.Messages
        .SelectMany(m => m.Contents)
        .OfType<TextReasoningContent>()
        .Select(rc => rc.Text)
        .Where(t => !string.IsNullOrWhiteSpace(t)));

// Render the reasoning text as Markdown
("### **Reasoning Content:** ").DisplayAs("text/markdown");
reasoningText.DisplayAs("text/markdown");

// Render the response string as Markdown
("### **Decision Reommendation:** ").DisplayAs("text/markdown");
chatLuxuryVacationDecisionImprovedWithGatheredIntelligenceResponseText.DisplayAs("text/markdown");

### **Reasoning Content:** 


*   Decision Framework: Ben Franklin Decision Framework.
    *   Question: "Should I take a luxury family vacation?"
    *   Context/Intelligence provided:
        1.  Considering secondary home investment.
        2.  Working long hours, no vacation in >1 year.
        3.  Recent promotion (pay raise + large bonus coming).
        4.  Car lease ending soon (need replacement).
        5.  Kids starting college soon.
        6.  Uncertainty regarding medical expense coverage.

    *   Step 1: Frame decision (Yes/No).
    *   Step 2 & 3: Create "Pro" and "Con" columns.
    *   Step 4: List max 5 reasons per side based *only* on provided intelligence. No filler.
    *   Step 5 & 6: Weighting/Canceling out (Crossing off). Note: The prompt asks to determine relative importance by crossing off similar values.
    *   Step 7: Recommendation is the side with the most remaining reasons.

    *   Decision: Should I take a luxury family vacation? (Yes or No)

    *   Pro Reasons (Arguments for "Yes"):
        1.  Burnout/Need for rest (Working long hours, no vacation in >1 year).
        2.  Increased disposable income (Recent promotion/pay raise).
        3.  Upcoming windfall (Large bonus coming in a few months).

    *   Con Reasons (Arguments for "No"):
        1.  Capital allocation conflict: Secondary home investment (Competing financial goal).
        2.  Upcoming major expense: Car replacement (Lease ending soon).
        3.  Upcoming major expense: College tuition (Kids starting college soon).
        4.  Financial uncertainty: Medical expenses (Unsure if insurance covers them).

    *   *Note on Weighting:* I need to balance these.
    *   Pro 1 (Burnout) is a high-value "need". Let's call it High weight.
    *   Pro 2 + Pro 3 (Promotion/Bonus) = Medium/High weight financial capability.
    *   Con 1 (Secondary Home) = High weight financial goal.
    *   Con 2 (Car) = Medium weight expense.
    *   Con 3 (College) = High weight expense.
    *   Con 4 (Medical) = Medium/High weight uncertainty.

    *Let's try to cancel them out:*
    - Pro 1 (Burnout) vs Con 1 (Secondary Home)? Hard to balance directly, but let's say the *need* for rest is a strong driver.
    - Pro 2 + Pro 3 (New money) can offset Con 2 (Car) and Con 4 (Medical uncertainty).

    *Let's follow the "Crossing off" logic strictly:*
    - Let'/s pair:
        - Pro 1 (Burnout) vs Con 3 (College - high priority expense). [Crossed out]
        - Pro 2 + Pro 3 (Promotion/Bonus) vs Con 1 (Secondary Home investment). [Crossed out]
        - Remaining Pros: None.
        - Remaining Cons: Con 2 (Car), Con 4 (Medical uncertainty).

    *Wait, the prompt says "The side with the most remaining reasons is the option one should select".*
    If I cross them all out equally, I need to be careful not to leave zero on both sides if possible, or ensure a winner.

    Let's re-evaluate weights:
    Pro 1 (Burnout) - Weight: High
    Pro 2 + Pro 3 (Promotion/Bonus) - Weight: Medium

    Con 1 (Secondary Home) - Weight: High
    Con 2 (Car replacement) - Weight: Medium
    Con 3 (College) - Weight: High
    Con 4 (Medical uncertainty) - Weight: Medium

    *Execution of cancellation:*
    - Pro 1 (High) cancels Con 1 (High).
    - Pro 2 + Pro 3 (Medium) cancels Con 2 (Medium).
    - Remaining Pros: 0.
    - Remaining Cons: Con 3 (College), Con 4 (Medical uncertainty).

    *Wait, if I do that, "No" wins because it has more remaining reasons.*

    Let's try another way to see if "Yes" can win.
    Pro 1 (Burnout) - High
    Pro 2 + Pro 3 (Promotion/Bonus) - Medium

    Con 1 (Secondary Home) - Medium
    Con 2 (Car replacement) - Low
    Con 3 (College) - Medium
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 + Con 3 (Medium + Medium = High).
    - Pro 2 + Pro 3 (Medium) cancels Con 2 + Con 4 (Low + Low = Medium).
    - Remaining Pros: 0.
    - Remaining Cons: 0.

    *Let's try to make "Yes" win by making the Pro side more robust.*
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Pay raise) - Medium
    Pro 3 (Bonus) - Low

    Con 1 (Secondary Home) - High
    Con 2 (Car replacement) - Medium
    Con 3 (College) - Medium
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 (High).
    - Pro 2 + Pro 3 (Medium + Low = Medium) cancels Con 2 (Medium).
    - Remaining Pros: 0.
    - Remaining Cons: Con 3 (Medium), Con 4 (Low).

    *Wait, the prompt says "The side with the most remaining reasons is the option one should select".*
    If I have 0 Pro and 2 Con, No wins.
    If I have 1 Pro and 0 Con, Yes wins.

    Let's look at the intelligence again.
    Pros:
    1. Burnout (High)
    2. Promotion/Raise (Medium)
    3. Bonus (Low)

    Cons:
    1. Secondary Home (High)
    2. Car replacement (Medium)
    3. College (Medium)
    4. Medical uncertainty (Low)

    Let's pair them to leave a Pro remaining.
    - Con 1 (High) cancels Pro 1 (High).
    - Con 2 (Medium) cancels Pro 2 (Medium).
    - Con 3 (Medium) - No pro left to cancel it.
    - Con 4 (Low) cancels Pro 3 (Low).

    Wait, if I do that, I have 0 Pros and 1 Con. "No" wins.

    Let's try:
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Raise) - Medium
    Pro 3 (Bonus) - Medium

    Con 1 (Secondary Home) - High
    Con 2 (Car replacement) - Low
    Con 3 (College) - Medium
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 (High).
    - Pro 2 (Medium) cancels Con 3 (Medium).
    - Pro 3 (Medium) cancels Con 2 + Con 4 (Low + Low = Medium).
    - Remaining: 0 vs 0.

    Let's try to make "Yes" win.
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Raise) - Medium
    Pro 3 (Bonus) - Low

    Con 1 (Secondary Home) - Medium
    Con 2 (Car replacement) - Low
    Con 3 (College) - High
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 3 (High).
    - Pro 2 (Medium) cancels Con 1 (Medium).
    - Pro 3 (Low) cancels Con 2 (Low).
    - Remaining: 0 Pros, 1 Con (Medical uncertainty). "No" wins.

    Let's try to make "Yes" win by making the Pro side very heavy.
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Raise) - Medium
    Pro 3 (Bonus) - Low

    Con 1 (Secondary Home) - Low
    Con 2 (Car replacement) - Low
    Con 3 (College) - High
    Con 4 (Medical uncertainty) - Medium

    *Cancellation:*
    - Pro 1 (High) cancels Con 3 (High).
    - Pro 2 (Medium) cancels Con 4 (Medium).
    - Pro 3 (Low) cancels Con 1 (Low).
    - Remaining: 0 Pros, 1 Con (Car replacement). "No" wins.

    *Wait, can I make a Pro win?*
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Raise) - Medium
    Pro 3 (Bonus) - Low

    Con 1 (Secondary Home) - Medium
    Con 2 (Car replacement) - Low
    Con 3 (College) - Low
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 (Medium) + Con 2 (Low) + Con 3 (Low) = High.
    - Pro 2 (Medium) cancels Con 4 (Low)? No, that doesn't work.
    - Let's try:
        Pro 1 (Burnout - High)
        Pro 2 (Promotion/Raise - Medium)
        Pro 3 (Bonus - Low)

        Con 1 (Secondary Home - Medium)
        Con 2 (Car replacement - Low)
        Con 3 (College - Low)
        Con 4 (Medical uncertainty - Low)

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 + Con 2 + Con 3 (Medium+Low+Low = High).
    - Pro 2 (Medium) cancels Con 4 (Low)? No.
    - Let's try:
        Pro 1 (Burnout - High)
        Pro 2 (Promotion/Raise - Medium)
        Pro 3 (Bonus - Low)

        Con 1 (Secondary Home - High)
        Con 2 (Car replacement - Low)
        Con 3 (College - Low)
        Con 4 (Medical uncertainty - Low)

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 (High).
    - Pro 2 (Medium) cancels Con 2 + Con 3 (Low + Low = Medium).
    - Pro 3 (Low) cancels Con 4 (Low).
    - Remaining: 0 Pros, 0 Cons.

    *Let's try to make "Yes" win.*
    Pro 1 (Burnout - High)
    Pro 2 (Promotion/Raise - Medium)
    Pro 3 (Bonus - Low)

    Con 1 (Secondary Home - Medium)
    Con 2 (Car replacement - Low)
    Con 3 (College - Low)
    Con 4 (Medical uncertainty - Low)

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 + Con 2 + Con 3 (Medium+Low+Low = High).
    - Pro 2 (Medium) is left.
    - Pro 3 (Low) cancels Con 4 (Low).
    - Remaining: 1 Pro (Pro 2), 0 Cons. "Yes" wins.

    *Wait, I must be careful not to invent weights that aren't logical.*
    Is it logical that the Secondary Home is Medium weight? Yes.
    Is it logical that Car/College are Low weight compared to a house? Yes.
    Is it logical that Burnout is High weight? Yes.

    Let's refine:
    Pro 1 (Burnout) - Weight: High
    Pro 2 (Promotion/Raise) - Weight: Medium
    Pro 3 (Bonus) - Weight: Low

    Con 1 (Secondary Home) - Weight: Medium
    Con 2 (Car replacement) - Weight: Low
    Con 3 (College) - Weight: Low
    Con 4 (Medical uncertainty) - Weight: Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 + Con 2 + Con 3 (Medium + Low + Low = High).
    - Pro 2 (Medium) remains.
    - Pro 3 (Low) cancels Con 4 (Low).
    - Result: Pro 2 is the only remaining reason. Decision: Yes.

    *Wait, let's check if there's a more natural way.*
    The prompt says "The side with the most remaining reasons is the option one should select".
    If I have 1 Pro and 0 Cons, "Yes" wins.

    Let's try to make "No" win (it feels more realistic given the financial obligations).
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Raise) - Medium
    Pro 3 (Bonus) - Low

    Con 1 (Secondary Home) - High
    Con 2 (Car replacement) - Medium
    Con 3 (College) - Medium
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 (High).
    - Pro 2 (Medium) cancels Con 2 (Medium).
    - Pro 3 (Low) cancels Con 4 (Low).
    - Remaining: Con 3 (College - Medium).
    - Result: "No" wins.

    This feels more balanced and uses all the data points without being arbitrary.

    *Final Plan:*
    Decision: Should I take a luxury family vacation?
    Pro Side:
    1. Burnout/Need for rest (High)
    2. Recent promotion/pay raise (Medium)
    3. Upcoming large bonus (Low)

    Con Side:
    1. Secondary home investment (High)
    2. Car replacement needs (Medium)
    3. College tuition upcoming (Medium)
    4. Medical expense uncertainty (Low)

    *Cancellation Process:*
    - Pro 1 (High) cancels Con 1 (High).
    - Pro 2 (Medium) cancels Con 2 (Medium).
    - Pro 3 (Low) cancels Con 4 (Low).
    - Remaining: Con 3.
    - Decision: No.

    *Wait, let me check the prompt instructions again.*
    "The side with the most remaining reasons is the option one should select".
    If I have 0 Pros and 1 Con, "No" wins.

    Let's double check if there are any other ways to interpret "crossing off".
    Usually, in Ben Franklin's method, you cross out a Pro with a Con of equal weight.
    If I cross them all out, the one that *can't* be crossed out is the winner.

    Let's try to make "Yes" win just to see if it's possible without being too weird.
    Pro 1 (Burnout) - High
    Pro 2 (Promotion/Raise) - Medium
    Pro 3 (Bonus) - Low

    Con 1 (Secondary Home) - Medium
    Con 2 (Car replacement) - Low
    Con 3 (College) - Low
    Con 4 (Medical uncertainty) - Low

    *Cancellation:*
    - Pro 1 (High) cancels Con 1 + Con 2 + Con 3 (Medium+Low+Low = High).
    - Pro 2 (Medium) remains.
    - Pro 3 (Low) cancels Con 4 (Low).
    - Winner: Yes.

    Actually, the "No" version is more logically sound because College and Car replacement are significant financial weights. Let's stick to the "No" version as it feels like a better application of "reasoning through uncertainty/tradeoffs".

    *Wait*, I need to present this in a Markdown table.
    The prompt says: "Format the response using only a Markdown table. Only return a Markdown table."

    Table structure:
    Column 1: Pro (Arguments For)
    Column 2: Con (Arguments Against)

    I will include the weights or the cancellation status in the table cells to show the work, as I cannot use text outside the table. Or I can just list them and let the logic be clear. But the prompt asks me to "Apply the following steps".
    If I only return a table, how do I show the "Decision"?
    I will include the decision in the table (perhaps as a final row or a header-like cell).

    Let's refine the Table content:
    Row 1: Pro | Con
    Row 2: Burnout (High) [Crossed Out] | Secondary Home Investment (High) [Crossed Out]
    Row 3: Promotion/Raise (Medium) [Crossed Out] | Car Replacement (Medium) [Crossed Out]
    Row 4: Upcoming Bonus (Low) [Crossed Out] | College Tuition (Medium)
    Row 5: | Medical Expense Uncertainty (Low) [Crossed Out]
    Row 6: FINAL DECISION: NO |

    Wait, the prompt says "Do not use any headings higher than ###". It also says "Format the response using only a Markdown table. Only return a Markdown table."
    This means I cannot have a header outside the table.
    I will put the decision in the table.

    Let's re-verify the logic for "No":
    Pro:
    1. Burnout (High)
    2. Promotion/Raise (Medium)
     3. Bonus (Low)

    Con:
    1. Secondary Home (High)
    2. Car Replacement (Medium)
    3. College Tuition (Medium)
    4. Medical Uncertainty (Low)

    Cancellation:
    Pro 1 vs Con 1 -> Both gone.
    Pro 2 vs Con 2 -> Both gone.
    Pro 3 vs Con 4 -> Both gone.
    Remaining: Con 3.
    Decision: No.

    Wait, if I have "Con 3" left, the side with the most remaining reasons is "Con".
    The decision is "No".

    Let's check the rules for the table again.
    "Format the response using only a Markdown table. Only return a Markdown table."
    I will make a two-column table.

    Column 1: Pro (Arguments For)
    Column 2: Con (Arguments Against)

    Rows:
    Row 1: Burnout [High] | Secondary Home Investment [High]
    Row 2: Promotion/Raise [Medium] | Car Replacement [Medium]
    Row 3: Upcoming Bonus [Low] | College Tuition [Medium]
    Row 4: | Medical Expense Uncertainty [Low]
    Row 5: **DECISION: NO** |

    Wait, I can't use "Crossed Out" text easily in a table without it looking messy. I will use strikethrough `~~text~~`.

    Let's check the constraints one more time.
    - Max 5 reasons per side. (I have 3 Pro, 4 Con).
    - No filler.
    - Decision based on most remaining reasons.
    - Only return a Markdown table.

    One detail: "The side with the most remaining reasons is the option one should select".
    If I cross out everything except Con 3, then Con has 1 reason and Pro has 0. So No wins.

    Let's try to make it cleaner.
    Pro:
    - ~~Burnout (High)~~
    - ~~Promotion/Raise (Medium)~~
    - ~~Upcoming Bonus (Low)~~

    Con:
    - ~~Secondary Home Investment (High)~~
    - ~~Car Replacement (Medium)~~
    - College Tuition (Medium)
    - ~~Medical Expense Uncertainty (Low)~~

    Final Decision: No.

    Wait, I'll put the decision in a single cell at the bottom or as a separate row.

    One more check on "Weighting":
    "This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half."
    If I cross out Pro 1 with Con 1, they are both gone.
    If I cross out Pro 2 with Con 2, they are both gone.
    If I cross out Pro 3 with Con 4, they are both gone.
    Remaining: Con 3.

    This is a valid application of the framework.

    Final Table Structure:
    | Pro (Arguments For) | Con (Arguments Against) |
    | :--- | :--- |
    | ~~Burnout/Need for Rest (High Weight)~~ | ~~Secondary Home Investment (High Weight)~~ |
    | ~~Promotion/Pay Raise (Medium Weight)~~ | ~~Car Replacement (Medium Weight)~~ |
    | ~~Upcoming Large Bonus (Low Weight)~~ | College Tuition (Medium Weight) |
    | | ~~Medical Expense Uncertainty (Low Weight)~~ |
    | **DECISION: NO** | |

    Wait, the prompt says "Do not use any headings higher than ###". Since I am only returning a table, there are no headings.
    "Only return a Markdown table." - This is key.

    Let's ensure all intelligence is used correctly.
    1) Secondary home (Con 1) - Used.
    2) Long hours/No vacation (Pro 1) - Used.
    3) Promotion/Raise (Pro 2) - Used.
    4) Bonus (Pro 3) - Used.
    5) Car lease ending (Con 2) - Used.
    6) Kids college (Con 3) - Used.
    7) Medical expenses (Con 4) - Used.

    All 6 points of intelligence are used.

    One small check: "The side with the most remaining reasons is the option one should select".
    If Pro has 0 and Con has 1, then Con wins. The decision is No.

    Wait, I'll make sure the table looks professional.

    | Pro (Arguments For) | Con (Arguments Against) |
    | :--- | :--- |
    | ~~Burnout/Need for Rest (High Weight)~~ | ~~Secondary Home Investment (High Weight)~~ |
    | ~~Promotion/Pay Raise (Medium Weight)~~ | ~~Car Replacement (Medium Weight)~~ |
    | ~~Upcoming Large Bonus (Low Weight)~~ | College Tuition (Medium Weight) |
    | | ~~Medical Expense Uncertainty (Low Weight)~~ |
    | **FINAL DECISION: NO** | |

    Wait, I should check if "College Tuition" is a "reason or argument". Yes.
    Is "No" the decision? Yes.

    One last look at Step 6/7.
    Step 6: Determine relative importance... crossing them off.
    Step 7: Side with most remaining reasons.

    If I cross out Pro 1 and Con 1, they are gone.
    If I cross out Pro 2 and Con 2, they are gone.
    If I cross out Pro 3 and Con 4, they are gone.
    Only Con 3 remains.
    Con has 1 reason remaining. Pro has 0.
    Therefore, the decision is No.

    Wait, can I make "Yes" win?
    Pro: Burnout (High), Promotion (Medium), Bonus (Low)
    Con: Home (High), Car (Low), College (Low), Medical (Low)
    Cross out Pro 1 with Con 1.
    Cross out Pro 2 with Con 2 + Con 3.
    Cross out Pro 3 with Con 4.
    Remaining: 0 vs 0.

    If I want "Yes" to win, I need a Pro that doesn't get cancelled.
    Pro: Burnout (High), Promotion (Medium), Bonus (Low)
    Con: Home (High), Car (Low), College (Low), Medical (Low)
    Wait, if I cross out Pro 1 with Con 1...
    If I cross out Pro 2 with Con 2...
    If I cross out Pro 3 with Con 3...
    Then Con 4 is left. "No" wins.

    It seems in almost all logical weightings where I use all the data, "No" or a tie occurs because there are more "Con" items (4) than "Pro" items (3). To have a "Yes" win, I would need to cancel out all 4 Cons using only 3 Pros. This is possible if one Pro is very heavy.

    Let'/s try:
    Pro 1 (High)
    Pro 2 (Medium)
    Pro 3 (Low)

    Con 1 (Medium)
    Con 2 (Low)
    Con 3 (Low)
    Con 4 (Low)

    Cancellation:
    Pro 1 (High) cancels Con 1 + Con 2 + Con 3 (Med+Low+Low = High).
    Pro 2 (Medium) remains.
    Pro 3 (Low) cancels Con 4 (Low).
    Winner: Pro 2 (Yes).

    However, the "No" version is more grounded in the actual weight of the items provided (College and Home are usually higher priority than a car or medical uncertainty). I will stick to the "No" version as it's more analytically sound.

    Final check on formatting:
    - No headings > ###. (None used)
    - No horizontal lines. (None used)
    - Only Markdown table. (Yes)
    - Max 5 reasons per side. (3 Pro, 4 Con - Yes)
    - Decision based on most remaining reasons. (Yes)

    One detail: The prompt says "Do not use any headings higher than ###". Since I'm only returning a table, I don't have headers. But the table itself has a header row. That is part of the Markdown table syntax, not a Markdown heading.

    Final Table Content Construction:
    Pro Column:
    1. ~~Burnout/Need for Rest (High)~~
    2. ~~Promotion/Pay Raise (Medium)~~
    3. ~~Upcoming Large Bonus (Low)~~

    Con Column:
    1. ~~Secondary Home Investment (High)~~
    2. ~~Car Replacement (Medium)~~
    3. College Tuition (Medium)
    4. ~~Medical Expense Uncertainty (Low)~~

    Decision: NO.

    Wait, I'll check the "Pro" count again.
    1. Burnout/Rest
    2. Promotion/Raise
    3. Bonus
    Total 3.

    Con Count:
    1. Secondary Home
    2. Car Replacement
    3. College Tuition
    4. Medical Uncertainty
    Total 4.

    If I cancel them out as planned, Con has 1 left. Decision is No.

### **Decision Reommendation:** 

| Pro (Arguments For) | Con (Arguments Against) |
| :--- | :--- |
| ~~Burnout/Need for Rest (High Weight)~~ | ~~Secondary Home Investment (High Weight)~~ |
| ~~Promotion/Pay Raise (Medium Weight)~~ | ~~Car Replacement (Medium Weight)~~ |
| ~~Upcoming Large Bonus (Low Weight)~~ | College Tuition (Medium Weight) |
| | ~~Medical Expense Uncertainty (Low Weight)~~ |
| **FINAL DECISION: NO** | |

---
### Step 7 - Open Source AI & Commercial AI Providers  

In .NET you can include mutliple AI chatclients with different service providers. This allows for hybrid workflows from a single service provider:
* Capability Optimizations: Use SLMs for domain specific tasks and LLMs for complex decision reasoning
* Decision Optimizations: Apply an (ensemble) decision self-consitency pattern  
* Capacity Optimizations: Splitting functions, plugins, personas or agents across different AI services

In [ ]:
// Import the required NuGet configuration packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.6"
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.6"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using Microsoft.Extensions.DependencyInjection;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.IO;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// Retrieve the configuration settings for the Azure OpenAI service
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];

// Retrieve the configuration settings for the local OpenAI-compatible service
var localApiKey = "not_needed_for_lmstudio_ollama_etc"; // local LLMs do not require an API key
var localUrl = "http://10.0.0.61:1234/v1/"; // this can be localhost or an IP address
var localModelName = "google/gemma-4-26b-a4b"; // Another Option: "openai/gpt-oss-20b";

// Create a service collection for dependency injection
var services = new ServiceCollection();

// Add Cloud AI Client Configuration
var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);
var cloudChatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();

// Add Local OpenAI Client Configuration
var apiCredentials = new ApiKeyCredential(localApiKey);
var openAIClientOptions = new OpenAIClientOptions
{
    Endpoint = new Uri(localUrl)
};
// Create a local AI client 
var localAIClient = new OpenAIClient(apiCredentials, openAIClientOptions);

// Wrap the OpenAI-compatible chat client with the Microsoft.Extensions.AI abstraction
IChatClient localAIChatClient = localAIClient.GetChatClient(localModelName)
    .AsIChatClient();

// Add cloud and local AI clients to the service collection for dependency injection
services.AddKeyedSingleton<IChatClient>("cloudAI", cloudChatClient);
services.AddKeyedSingleton<IChatClient>("localAI", localAIChatClient);

var provider = services.BuildServiceProvider();

// Reference the cloud and local AI clients from the service provider
var cloud = provider.GetRequiredKeyedService<IChatClient>("cloudAI");
var local = provider.GetRequiredKeyedService<IChatClient>("localAI");

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Microsoft.Extensions.AI.OpenAI, 10.5.0 Microsoft.Extensions.Configuration.Json, 10.0.6 Microsoft.Extensions.DependencyInjection, 10.0.6 OpenAI, 2.10.0